# 🏥 MedImage — Template Notebook

Template สำหรับเข้าถึงระบบ MedImage จาก Jupyter

| Section | เนื้อหา |
|---|---|
| 1 | Install & Import |
| 2 | MinIO — อ่าน/เขียนไฟล์ |
| 3 | Ray Cluster — ส่ง Job |
| 4 | Models — โหลด model จาก Jobs API |
| 5 | Utilities |
| 6 | Modal.com — GPU Cluster & Deploy |

---
## 1 · Install & Import

In [ ]:
# ติดตั้ง library ที่จำเป็น (รันครั้งแรกครั้งเดียว)
import subprocess, sys

pkgs = ["boto3", "s3fs", "ray[default]", "pandas", "matplotlib", "requests"]
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet"] + pkgs)

In [ ]:
import boto3
import s3fs
import requests
import pandas as pd
import json
from pathlib import Path

# ─── Config ───────────────────────────────────────────────────────────────────
MINIO_URL        = 'http://minio:9000'
MINIO_ACCESS_KEY = 'minioadmin'
MINIO_SECRET_KEY = 'minioadmin'

RAY_HEAD_IP      = '100.68.53.118'
RAY_DASHBOARD    = f'http://{RAY_HEAD_IP}:8265'
RAY_CLIENT_URL   = f'ray://{RAY_HEAD_IP}:10001'

# GPU fraction ต่อ 1 task: 0.5 = รัน 2 task พร้อมกันต่อ 1 GPU, 1.0 = exclusive
GPU_FRACTION     = 0.5


BACKEND_URL      = 'http://medimage-api:8000'print('✅ Config loaded')


---
## 2 · MinIO — อ่าน/เขียนไฟล์

In [ ]:
# เชื่อมต่อ MinIO ผ่าน boto3
s3 = boto3.client(
    's3',
    endpoint_url=MINIO_URL,
    aws_access_key_id=MINIO_ACCESS_KEY,
    aws_secret_access_key=MINIO_SECRET_KEY,
    region_name='us-east-1',
)

# list buckets
buckets = [b['Name'] for b in s3.list_buckets()['Buckets']]
print('Buckets:', buckets)

In [ ]:
# List ไฟล์ใน bucket
BUCKET = 'medimage-raw'   # เปลี่ยน bucket ได้

resp = s3.list_objects_v2(Bucket=BUCKET)
files = [obj['Key'] for obj in resp.get('Contents', [])]
print(f'{BUCKET}: {len(files)} files')
for f in files[:20]:
    print(' ', f)

In [ ]:
# Download ไฟล์
FILE_KEY  = 'example.png'           # path ใน bucket
LOCAL_DST = '/home/jovyan/work/example.png'

s3.download_file(BUCKET, FILE_KEY, LOCAL_DST)
print(f'Downloaded → {LOCAL_DST}')

In [ ]:
# Upload ไฟล์
LOCAL_SRC  = '/home/jovyan/work/result.csv'
BUCKET_DST = 'medimage-raw'
KEY_DST    = 'outputs/result.csv'

s3.upload_file(LOCAL_SRC, BUCKET_DST, KEY_DST)
print(f'Uploaded → s3://{BUCKET_DST}/{KEY_DST}')

In [ ]:
# อ่าน CSV จาก MinIO โดยตรงด้วย s3fs + pandas
fs = s3fs.S3FileSystem(
    endpoint_url=MINIO_URL,
    key=MINIO_ACCESS_KEY,
    secret=MINIO_SECRET_KEY,
)

# df = pd.read_csv(fs.open('medimage-raw/labels.csv'))
# df.head()

# list ไฟล์ทั้งหมดด้วย s3fs
print(fs.ls('medimage-raw/'))

---
## 3 · Ray Cluster — ส่ง Job

In [ ]:
# ตรวจสอบสถานะ Ray Cluster ผ่าน Dashboard REST API
resp = requests.get(f'{RAY_DASHBOARD}/api/cluster_status', timeout=5)
if resp.ok:
    data = resp.json()
    print('Ray cluster status:', data.get('status', data))
else:
    print('Dashboard status:', resp.status_code)

In [ ]:
# ดู nodes + GPU ใน cluster
resp = requests.get(f'{RAY_DASHBOARD}/nodes?view=summary', timeout=5)
if resp.ok:
    nodes = resp.json().get('data', {}).get('summary', [])
    for n in nodes:
        gpus = n.get('gpus', [])
        print(f"Node: {n['raylet']['nodeManagerAddress']}  "
              f"CPU: {n['cpus'][1]:.0f}  "
              f"GPU: {len(gpus)}  "
              f"State: {n['raylet']['state']}")
else:
    print(resp.status_code)

In [ ]:
# เชื่อม Ray Client จาก Jupyter
import ray

if not ray.is_initialized():
    ray.init(RAY_CLIENT_URL, ignore_reinit_error=True)

print(ray.cluster_resources())

In [ ]:
# รัน Remote Function บน Ray Cluster
# num_gpus=GPU_FRACTION → เช่น 0.5 = 1 GPU รัน task พร้อมกันได้ 2 task (ไม่ต้องรอคิว)
# num_gpus=1.0         → exclusive: 1 task ต่อ 1 GPU ทั้งใบ
# num_gpus=0.0         → ไม่ใช้ GPU (CPU only)
@ray.remote(num_gpus=GPU_FRACTION)
def train_step(batch_id: int) -> dict:
    import time, random
    gpu_ids = ray.get_gpu_ids()   # ดู GPU ที่ Ray จัดให้
    time.sleep(1)
    return {'batch': batch_id, 'loss': random.random(), 'gpu_ids': gpu_ids}


# ส่ง 4 batch แบบ parallel — ถ้า GPU_FRACTION=0.5 และมี 2 GPU จะรันพร้อมกัน 4 taskprint(results)

futures = [train_step.remote(i) for i in range(4)]results = ray.get(futures)

In [ ]:
# ส่ง Ray Job ผ่าน Jobs API (fire-and-forget)
from ray.job_submission import JobSubmissionClient

client = JobSubmissionClient(RAY_DASHBOARD)

job_id = client.submit_job(
    entrypoint='python train.py --epochs 10 --model efficientnet-b2',
    entrypoint_num_cpus=4,
    entrypoint_num_gpus=GPU_FRACTION,    # สัดส่วน GPU สำหรับ entrypoint process
    runtime_env={
        'working_dir': '/home/jovyan/work',
        'pip': ['torch', 'torchvision', 'timm'],
        'env_vars': {

            # เปิด CUDA MPS สำหรับ hardware-level GPU sharing (optional)print('Submitted job:', job_id)

            # 'CUDA_MPS_ACTIVE_THREAD_PERCENTAGE': str(int(GPU_FRACTION * 100)),)

        },    },

In [ ]:
# ดูสถานะ Job
# job_id = 'raysubmit_XXXX'   # ใส่ job_id จาก cell บน
status = client.get_job_status(job_id)
logs   = client.get_job_logs(job_id)
print('Status:', status)
print('Logs:', logs[-500:] if len(logs) > 500 else logs)

---
## 4 · Models — โหลด model จากระบบ

In [ ]:
# ดึงรายการ Model จาก MedImage Backend (เฉพาะ job ที่ completed)
resp = requests.get(f'{BACKEND_URL}/api/jobs', params={'view': 'models'})
jobs = resp.json()

df_models = pd.DataFrame([{
    'id':          j['id'],
    'name':        j.get('name', j['id'][:8]),
    'model_name':  j.get('model_name'),
    'engine':      j.get('engine'),
    'epochs':      j.get('epochs'),
    'finished_at': j.get('finished_at'),
} for j in jobs])

print(f'Models: {len(df_models)}')
df_models

In [ ]:
# ดูรายละเอียด + log ของ model ที่เลือก
JOB_ID = df_models.iloc[0]['id']   # เปลี่ยนเป็น id ที่ต้องการ

resp = requests.get(f'{BACKEND_URL}/api/jobs/{JOB_ID}')
job  = resp.json()

print('Name      :', job.get('name'))
print('Model     :', job.get('model_name'))
print('Engine    :', job.get('engine'))
print('Status    :', job.get('status'))
print('Progress  :', job.get('progress'), '%')
print('\n--- Training Log ---')
print(job.get('log', '')[-1000:])

In [ ]:
# โหลด PyTorch model จาก MinIO (ถ้า checkpoint ถูก save ไว้)
import torch

MODEL_KEY  = f'models/{JOB_ID}/best.pt'   # path ใน MinIO
LOCAL_PATH = f'/home/jovyan/work/best_{JOB_ID[:8]}.pt'

try:
    s3.download_file('medimage-raw', MODEL_KEY, LOCAL_PATH)
    model = torch.load(LOCAL_PATH, map_location='cpu')
    model.eval()
    print('Model loaded:', type(model))
except Exception as e:
    print('Error:', e)

In [ ]:
# Inference บน DICOM / PNG image
from PIL import Image
import torchvision.transforms as T
import numpy as np

transform = T.Compose([
    T.Resize((640, 640)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225]),
])

# image_path = '/home/jovyan/work/example.png'   # เปลี่ยน path
# img = Image.open(image_path).convert('RGB')
# tensor = transform(img).unsqueeze(0)            # [1, 3, H, W]

# with torch.no_grad():
#     output = model(tensor)
#     probs  = torch.softmax(output, dim=1)
#     pred   = probs.argmax(dim=1).item()
#     print(f'Prediction: class {pred}  confidence: {probs[0][pred]:.3f}')

---
## 5 · Utilities

In [ ]:
# ปิด Ray Client เมื่อเสร็จงาน
# ray.shutdown()

---
## 6 · Modal.com — GPU Cluster & Deploy

ใช้ GPU cloud ของ [Modal.com](https://modal.com) ผ่าน Backend API:

1. ตั้ง credentials ที่หน้า **Modal Config** ใน UI (token_id + token_secret) — ทำครั้งเดียว
2. **Start Cluster** — spin up Ray cluster บน Modal (เลือก GPU type / จำนวน worker)
3. ส่ง train job ผ่าน Ray (Section 3) หรือ deploy model ที่ train เสร็จแล้ว (Section 4)
4. **Stop Cluster** เมื่อใช้งานเสร็จ เพื่อหยุดคิดค่าใช้จ่าย

**GPU types ที่รองรับ**: `T4`, `L4`, `A10G`, `L40S`, `A100 (40GB)`, `A100-80GB`, `H100`, `cpu`

In [ ]:
# ตรวจสอบว่า Backend มี Modal credentials อยู่หรือไม่
# (ตั้งค่าที่ UI: Modal Configuration → Save Credentials)
resp = requests.get(f'{BACKEND_URL}/api/modal/credentials', timeout=5)
creds = resp.json()
if creds.get('configured'):
    print(f"✅ Credentials OK — token_id: {creds['token_id']}")
    print(f"   updated_at: {creds.get('updated_at')}")
else:
    print('❌ ยังไม่ได้ตั้ง credentials → ไปที่ UI: Modal Configuration แล้วใส่ token_id + token_secret')
    print('   สร้าง token ได้ที่ https://modal.com/settings')

In [ ]:
# ทดสอบ credentials ด้วย `modal profile current`
resp = requests.post(f'{BACKEND_URL}/api/modal/verify', timeout=30)
v = resp.json()
print('ok:    ', v.get('ok'))
print('output:', (v.get('output') or '').strip()[:400])

In [ ]:
# Start Modal Ray Cluster
# gpu_type: T4 | L4 | A10G | L40S | A100 | A100-80GB | H100 | cpu
# num_workers: จำนวน worker (head node ถูกนับเป็น worker แรก)
MODAL_GPU_TYPE    = 'T4'
MODAL_NUM_WORKERS = 1

resp = requests.post(
    f'{BACKEND_URL}/api/modal/start',
    json={'gpu_type': MODAL_GPU_TYPE, 'num_workers': MODAL_NUM_WORKERS},
    timeout=10,
)
print('Start:', resp.status_code, resp.json())
print('⏳ deploy ใช้เวลา 1–3 นาที — cell ถัดไปจะ poll สถานะให้')

In [ ]:
# Poll สถานะจนกว่า cluster จะ running (timeout ~3 นาที)
import time

MODAL_RAY_URL = None
for i in range(60):
    s = requests.get(f'{BACKEND_URL}/api/modal/status', timeout=10).json()
    status = s.get('status')
    print(f'[{i:>2}] status={status:<10} gpu={s.get("gpu_type")} workers={s.get("num_workers")} url={s.get("ray_url") or "-"}')
    if status == 'running':
        MODAL_RAY_URL = s.get('ray_url')
        break
    if status == 'error':
        print('Logs:', s.get('logs', []))
        break
    time.sleep(5)
else:
    print('⏱️ timeout — เช็ค logs ที่ UI')

In [ ]:
# Scale workers: +1 เพิ่ม worker, -1 ลด worker (เฉพาะ worker, head ไม่หยุด)
SCALE_DELTA = 1
resp = requests.post(
    f'{BACKEND_URL}/api/modal/scale',
    json={'delta': SCALE_DELTA},
    timeout=10,
)
print('Scale:', resp.status_code, resp.json())

In [ ]:
# ดูรายการ model ที่ deploy บน Modal (ต่อ model)
resp = requests.get(f'{BACKEND_URL}/api/modal/deployments', timeout=10)
deployments = resp.json().get('deployments', [])

df_dep = pd.DataFrame([{
    'id':          d.get('id'),
    'model_name':  d.get('model_name'),
    'modal_url':   d.get('modal_url'),
    'gpu_type':    d.get('gpu_type'),
    'num_workers': d.get('num_workers'),
    'status':      d.get('status'),
} for d in deployments])
print(f'Modal deployments: {len(df_dep)}')
df_dep

In [ ]:
# Deploy model ที่ train เสร็จแล้วไปยัง Modal
# (ต้องเลือก job ที่ status=completed และมี weights ใน MinIO)
DEPLOY_JOB_ID = None
if len(df_models) == 0:
    print('❌ ไม่มี model — train model ใน Section 4 ก่อน')
else:
    DEPLOY_JOB_ID = df_models.iloc[0]['id']
    resp = requests.post(
        f'{BACKEND_URL}/api/jobs/{DEPLOY_JOB_ID}/deploy-modal',
        json={'gpu_type': 'T4', 'num_workers': 1},
        timeout=10,
    )
    print(f'Deploy {DEPLOY_JOB_ID[:8]}: {resp.status_code} {resp.json()}')

In [ ]:
# Poll สถานะ deployment (cold-start 2–5 นาที)
MODAL_URL = None
if DEPLOY_JOB_ID:
    for i in range(60):
        s = requests.get(
            f'{BACKEND_URL}/api/jobs/{DEPLOY_JOB_ID}/deploy-modal/status',
            timeout=10,
        ).json()
        st = s.get('status')
        print(f'[{i:>2}] {st:<10} url={s.get("url") or "-"}')
        if st == 'running':
            MODAL_URL = s.get('url')
            break
        if st in ('error', 'idle'):
            print('Logs:', s.get('logs', [])[-5:])
            break
        time.sleep(5)
    print(f'\nMODAL_URL = {MODAL_URL}')

In [ ]:
# Inference ผ่าน Modal endpoint — POST /predict ด้วย multipart file
if not MODAL_URL:
    print('⏭️  ข้าม — ยังไม่มี MODAL_URL (deploy ยังไม่เสร็จ)')
else:
    SAMPLE_IMG = '/home/jovyan/work/example.png'
    if not Path(SAMPLE_IMG).exists():
        # ลองดาวน์โหลดจาก MinIO
        try:
            s3.download_file('medimage-raw', 'example.png', SAMPLE_IMG)
        except Exception as e:
            print(f'❌ หา {SAMPLE_IMG} ไม่เจอ: {e}')
            SAMPLE_IMG = None
    
    if SAMPLE_IMG and Path(SAMPLE_IMG).exists():
        with open(SAMPLE_IMG, 'rb') as f:
            r = requests.post(
                f'{MODAL_URL}/predict',
                files={'file': ('image.png', f, 'image/png')},
                timeout=60,
            )
        print('Status:', r.status_code)
        try:
            print('Result:', json.dumps(r.json(), indent=2, ensure_ascii=False))
        except Exception:
            print('Raw:', r.text[:500])

In [ ]:
# ตรวจ online + หยุด deployment เฉพาะ model นี้ (cluster ยังอยู่)
if DEPLOY_JOB_ID:
    s = requests.get(f'{BACKEND_URL}/api/jobs/{DEPLOY_JOB_ID}/modal-status', timeout=5).json()
    print('Online:', s.get('online'))
    
    # เอา comment ออกเพื่อหยุด deployment
    # resp = requests.post(f'{BACKEND_URL}/api/jobs/{DEPLOY_JOB_ID}/deploy-modal/stop', timeout=30)
    # print('Stop deploy:', resp.status_code, resp.json())

In [ ]:
# หยุด Modal Ray Cluster (release GPU → ลดค่าใช้จ่าย)
# เอา comment ออกเมื่อต้องการหยุด
# resp = requests.post(f'{BACKEND_URL}/api/modal/stop', timeout=10)
# print('Stop cluster:', resp.status_code, resp.json())
# print('⏳ รอ ~30s ให้ container drain แล้วเช็ค status อีกครั้ง')